In [ ]:
# Import libraries
import numpy as np
import pandas as pd

In [ ]:
# Sample dataset
student_data = {
    "Age": [20, 21, 22, 20, 23, 21],
    "StudyHours": [2.5, 4.0, 5.5, 3.0, 6.0, 4.5],
    "Gender": ["Female", "Male", "Female", "Male", "Female", "Male"],
    "City": ["Delhi", "Mumbai", "Delhi", "Chandigarh", "Mumbai", "Delhi"],
    "Education": [
        "School", "Undergraduate", "Postgraduate",
        "Undergraduate", "Postgraduate", "School"
    ],
    "Placed": ["No", "Yes", "Yes", "No", "Yes", "Yes"]
}

df = pd.DataFrame(student_data)
print(df)

In [ ]:
# Binary mapping
df_mapped = df.copy()

df_mapped["Gender"] = df_mapped["Gender"].map({
    "Female": 0,
    "Male": 1
})

df_mapped["Placed"] = df_mapped["Placed"].map({
    "No": 0,
    "Yes": 1
})

print(df_mapped[["Gender", "Placed"]])

In [ ]:
# Label encoding
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_label = df.copy()

df_label["CityCode"] = label_encoder.fit_transform(df_label["City"])

print(df_label[["City", "CityCode"]])
print("Classes:", label_encoder.classes_)

In [ ]:
# Ordinal encoding using mapping
education_order = {
    "School": 0,
    "Undergraduate": 1,
    "Postgraduate": 2
}

df_ordinal = df.copy()
df_ordinal["Education"] = df_ordinal["Education"].map(education_order)

print(df_ordinal[["Education"]])

In [ ]:
# One-hot encoding with Pandas
df_one_hot = pd.get_dummies(
    df,
    columns=["City"],
    dtype=int
)

print(df_one_hot)

In [ ]:
# One-hot encoding with drop_first
df_one_hot_reduced = pd.get_dummies(
    df,
    columns=["City"],
    drop_first=True,
    dtype=int
)

print(df_one_hot_reduced)

In [ ]:
# One-hot encoding with Scikit-Learn
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

city_encoded = encoder.fit_transform(df[["City"]])
city_columns = encoder.get_feature_names_out(["City"])

city_df = pd.DataFrame(
    city_encoded,
    columns=city_columns
)

print(city_df)

In [ ]:
# Standardisation
from sklearn.preprocessing import StandardScaler

sample = pd.DataFrame({
    "Age": [20, 25, 30, 35, 40],
    "Income": [250000, 350000, 500000, 700000, 900000]
})

standard_scaler = StandardScaler()
scaled_values = standard_scaler.fit_transform(sample)

standardised_df = pd.DataFrame(
    scaled_values,
    columns=sample.columns
)

print(standardised_df.round(3))

In [ ]:
# Min-Max normalisation
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
normalised_values = minmax_scaler.fit_transform(sample)

normalised_df = pd.DataFrame(
    normalised_values,
    columns=sample.columns
)

print(normalised_df)

In [ ]:
# Separate features and target
X = df.drop(columns=["Placed"])
y = df["Placed"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
# Check class distributions
print("Original distribution:")
print(y.value_counts(normalize=True))

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
# Correct scaling approach: split first, then fit only on training data
numeric_X = pd.DataFrame({
    "Age": [20, 21, 22, 20, 23, 21, 24, 25, 22, 23],
    "StudyHours": [2.5, 4.0, 5.5, 3.0, 6.0, 4.5, 7.0, 8.0, 5.0, 6.5]
})

target = pd.Series([0, 1, 1, 0, 1, 1, 1, 1, 0, 1])

X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(
    numeric_X,
    target,
    test_size=0.20,
    random_state=42,
    stratify=target
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

print(X_train_scaled)
print(X_test_scaled)

In [ ]:
# Dataset containing missing values for pipeline example
raw_data = {
    "Age": [20, 21, np.nan, 22, 23, 20, 24, 21],
    "StudyHours": [2.5, 4.0, 5.5, np.nan, 6.0, 3.0, 7.0, 4.5],
    "Gender": ["Female", "Male", "Female", "Male", None, "Male", "Female", "Male"],
    "City": ["Delhi", "Mumbai", "Delhi", "Chandigarh", "Mumbai", "Delhi", None, "Mumbai"],
    "Placed": [0, 1, 1, 0, 1, 0, 1, 1]
}

pipeline_df = pd.DataFrame(raw_data)

X = pipeline_df.drop(columns=["Placed"])
y = pipeline_df["Placed"]

print(pipeline_df)

In [ ]:
# Define numerical and categorical columns
numeric_features = ["Age", "StudyHours"]
categorical_features = ["Gender", "City"]

In [ ]:
# Numerical preprocessing pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
# Categorical preprocessing pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
# Combine preprocessing steps
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [ ]:
# Split and transform the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Prepared training shape:", X_train_ready.shape)
print("Prepared testing shape:", X_test_ready.shape)

In [ ]:
# Add Logistic Regression to the preprocessing pipeline
from sklearn.linear_model import LogisticRegression

complete_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

complete_model.fit(X_train, y_train)
predictions = complete_model.predict(X_test)

print("Predictions:", predictions)
print("Actual values:", y_test.to_numpy())

In [ ]:
# End-to-end preprocessing and classification example
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = {
    "Age": [20, 22, 21, np.nan, 24, 23, 20, 25, 22, 21],
    "StudyHours": [2, 5, 3, 4, 7, 6, np.nan, 8, 5, 3],
    "City": [
        "Delhi", "Mumbai", "Delhi", "Chandigarh", "Mumbai",
        "Delhi", "Chandigarh", "Mumbai", None, "Delhi"
    ],
    "Placed": [0, 1, 0, 1, 1, 1, 0, 1, 1, 0]
}

df_final = pd.DataFrame(data)

X = df_final.drop(columns=["Placed"])
y = df_final["Placed"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

numeric_features = ["Age", "StudyHours"]
categorical_features = ["City"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Predicted classes:", predictions)
print("Actual classes:", y_test.to_numpy())